In [1]:
# ==================================================
# SCRIPT DESCRIPTION (Markdown)
# ==================================================
"""
# EPC Heat Pump Data Mapping Script

This script processes building-level EPC and energy model outputs to:
- Match building folders with EPC metadata (HP_UPGRADE flag)
- Apply conditional column mapping rules
- Transfer energy values between CSV columns
- Support two workflows:
  (1) HP upgrade scenario (heat file → energy results file)
  (2) Non-upgrade scenario (energy results file → energy results file)
- Save updated energy results back to disk
"""

print("Script initialised.")

Script initialised.


In [2]:
# %%
# STEP 2 — IMPORT LIBRARIES
# --------------------------------------------------
import pandas as pd
from pathlib import Path

print("Libraries loaded.")

Libraries loaded.


In [3]:
# %%
# STEP 3 — LOAD EPC DATASET AND CREATE LOOKUP
# --------------------------------------------------
epc_path = Path("/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_hp_code.csv")

epc_df = pd.read_csv(epc_path)

# Ensure BUILDING_REFERENCE_NUMBER is string for matching folder names
epc_df["BUILDING_REFERENCE_NUMBER"] = epc_df["BUILDING_REFERENCE_NUMBER"].astype(str)

# Create lookup dictionary: {building_id: HP_UPGRADE}
hp_lookup = dict(zip(epc_df["BUILDING_REFERENCE_NUMBER"], epc_df["HP_UPGRADE"]))

print(f"EPC dataset loaded. {len(hp_lookup)} buildings mapped.")

EPC dataset loaded. 117 buildings mapped.


In [4]:
# %%
# STEP 4 — DEFINE BASE DIRECTORY AND GET BUILDINGS
# --------------------------------------------------
base_dir = Path("/Users/jakegehrung/Desktop/data_raw/baseline_models")

building_dirs = [d for d in base_dir.iterdir() if d.is_dir()]

print(f"Found {len(building_dirs)} building folders.")

Found 117 building folders.


In [5]:
# ==================================================
# STEP 5 — PROCESS BUILDINGS WITH SAFE ROW-ALIGNED MAPPING
# Description: Handles unequal row lengths safely (no alignment errors)
# ==================================================

success_count = 0
fail_count = 0
missing_epc_count = 0

for building_dir in building_dirs:

    building_id = building_dir.name
    print(f"\nProcessing building: {building_id}")

    # Check EPC lookup
    if building_id not in hp_lookup:
        print("No EPC entry found. Skipping.")
        missing_epc_count += 1
        continue

    hp_upgrade_flag = hp_lookup[building_id]

    heating_file = building_dir / "HEATING" / "hp_heat_by_fuel.csv"
    energy_file = building_dir / "TOTAL" / "energy_results.csv"

    if not heating_file.exists() or not energy_file.exists():
        print("Missing required files. Skipping.")
        fail_count += 1
        continue

    try:
        # LOAD DATA
        energy_df = pd.read_csv(energy_file)
        energy_df.columns = energy_df.columns.str.strip()

        # --------------------------------------------------
        # CASE 1 — INTRA-FILE COPY (energy → energy)
        # --------------------------------------------------
        if hp_upgrade_flag == 0:

            column_mapping = {
                "electricity_heat": "electricity_heat_hp",
                "lpg_heat": "lpg_heat_hp",
                "mineral_wood_heat": "mineral_wood_heat_hp",
                "mains_gas_heat": "mains_gas_heat_hp",
                "oil_heat": "oil_heat_hp",
                "wood_logs_heat": "wood_logs_heat_hp",
                "wood_pellets_heat": "wood_pellets_heat_hp",
                "wood_chips_heat": "wood_chips_heat_hp",
                "coal_heat": "coal_heat_hp"
            }

            missing_source = [c for c in column_mapping if c not in energy_df.columns]

            if missing_source:
                print(f"Missing source columns: {missing_source}")
                fail_count += 1
                continue

            for target_col in column_mapping.values():
                if target_col not in energy_df.columns:
                    energy_df[target_col] = pd.NA

            # SAFE ROW-BY-ROW COPY (handles length mismatch)
            n_rows = len(energy_df)

            for source_col, target_col in column_mapping.items():

                energy_df[target_col] = pd.NA  # reset cleanly

                energy_df.loc[:n_rows - 1, target_col] = (
                    energy_df[source_col]
                    .iloc[:n_rows]
                    .reset_index(drop=True)
                )

            energy_df.to_csv(energy_file, index=False)

            print("Success (HP_UPGRADE = 0, safe intra-file mapping)")
            success_count += 1

        # --------------------------------------------------
        # CASE 2 — HEAT FILE → ENERGY FILE
        # --------------------------------------------------
        elif hp_upgrade_flag == 1:

            column_mapping = {
                "electricity": "electricity_heat_hp",
                "lpg": "lpg_heat_hp",
                "mineral_wood": "mineral_wood_heat_hp",
                "mains_gas": "mains_gas_heat_hp",
                "oil": "oil_heat_hp",
                "wood_logs": "wood_logs_heat_hp",
                "wood_pellets": "wood_pellets_heat_hp",
                "wood_chips": "wood_chips_heat_hp",
                "coal": "coal_heat_hp"
            }

            heat_df = pd.read_csv(heating_file)

            heat_df.columns = heat_df.columns.str.strip()
            energy_df.columns = energy_df.columns.str.strip()

            missing_source = [c for c in column_mapping if c not in heat_df.columns]

            if missing_source:
                print(f"Missing source columns: {missing_source}")
                fail_count += 1
                continue

            for target_col in column_mapping.values():
                if target_col not in energy_df.columns:
                    energy_df[target_col] = pd.NA

            # SAFE ROW-BY-ROW COPY (handles 17519 vs 17520 mismatch)
            n_rows = len(energy_df)

            for source_col, target_col in column_mapping.items():

                energy_df[target_col] = pd.NA

                energy_df.loc[:n_rows - 1, target_col] = (
                    heat_df[source_col]
                    .iloc[:n_rows]
                    .reset_index(drop=True)
                )

            energy_df.to_csv(energy_file, index=False)

            print("Success (HP_UPGRADE = 1, safe heat → energy mapping)")
            success_count += 1

        else:
            print(f"Unexpected HP_UPGRADE value: {hp_upgrade_flag}")
            fail_count += 1

    except Exception as e:
        print(f"Error: {e}")
        fail_count += 1

print("\n--- SUMMARY ---")
print(f"Successful: {success_count}")
print(f"Failed: {fail_count}")
print(f"Missing EPC entries: {missing_epc_count}")


Processing building: 1001664924
Success (HP_UPGRADE = 0, safe intra-file mapping)

Processing building: 1001829085
Success (HP_UPGRADE = 1, safe heat → energy mapping)

Processing building: 1001063639
Success (HP_UPGRADE = 1, safe heat → energy mapping)

Processing building: 1001829071
Success (HP_UPGRADE = 0, safe intra-file mapping)

Processing building: 1234761002
Success (HP_UPGRADE = 1, safe heat → energy mapping)

Processing building: 1002539407
Success (HP_UPGRADE = 1, safe heat → energy mapping)

Processing building: 1001063637
Success (HP_UPGRADE = 0, safe intra-file mapping)

Processing building: 1001664941
Success (HP_UPGRADE = 0, safe intra-file mapping)

Processing building: 1001991633
Success (HP_UPGRADE = 1, safe heat → energy mapping)

Processing building: 1235057414
Success (HP_UPGRADE = 0, safe intra-file mapping)

Processing building: 1001829079
Success (HP_UPGRADE = 1, safe heat → energy mapping)

Processing building: 1001664922
Success (HP_UPGRADE = 0, safe intra-